## 💻 Ambiente de Execução e Preparação de Dados

O pipeline de extração, balanceamento e empacotamento dos datasets unificados foi executado em ambiente de nuvem, garantindo a reprodutibilidade e o manuseio eficiente de um grande volume de dados sem estourar limites locais de I/O.

**Especificações da Infraestrutura:**
* **Plataforma:** Kaggle Notebooks
* **Acelerador de Hardware:** NVIDIA Tesla T4 GPU (16GB VRAM)
* **Framework Base:** PyTorch, Pandas e PyArrow
* **Processamento:** CUDA (`cuda:0`) ativado para suporte e aceleração.

**Resumo do Dataset Unificado (Pós-Processamento):**
* **Treino:** 138.000 amostras balanceadas e embaralhadas (Real / Fake)
* **Validação:** 12.000 amostras (Real / Fake)
* **Inferência Física (Teste):** 20.000 amostras (10.000 do SID-Set + 10.000 do Artifact)

**Tempos de Processamento e Exportação (Blindada para ZIP):**
Os dados originais — provenientes do AIGI-Holmes, Artifact e arquivos Parquet do SID-Set — foram unificados, submetidos a compressão JPEG aleatória e exportados de forma contínua:

| Etapa de Exportação | Arquivo de Destino | Volume de Imagens | Tempo de Empacotamento |
| :--- | :--- | :---: | :--- |
| **Treino** | `dataset_treino_estatico.zip` | 138.000 | ~ 6h 07m |
| **Validação** | `dataset_treino_estatico.zip` | 12.000 | ~ 31m 28s |
| **Teste (Artifact)** | `dataset_inferencia_estatico.zip` | 10.000 | ~ 1m 54s |
| **Teste (SID-Set)** | `dataset_inferencia_estatico.zip` | 10.000 | ~ 1h 12m |


## **Importações, clone do repositório e Configurações Iniciais:**

In [1]:
import os
import sys
import gc
import io
import glob
import random
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/datasets_cache"

# Clone do repositório e instalação das dependências
!git clone https://github.com/Jvfrostbr/ClipBased-SyntheticImageDetection.git
!pip install hf_xet open_clip_torch

# Configuração dos caminhos do sistema para o Python enxergar a pasta 'networks'
%cd /kaggle/working/ClipBased-SyntheticImageDetection
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Importação do classificador do meu repositório
from networks.clip_custom_detector import CLIPImageClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Ambiente configurado com sucesso! Dispositivo: {DEVICE}")

#  Parâmetros iniciais dos datasets
SEED = 42
random.seed(SEED)
QTD_TREINO = 23000
QTD_VAL = 2000
QTD_INFERENCIA = 5000
QTD_TOTAL = QTD_TREINO + QTD_VAL + QTD_INFERENCIA  # 30.000 por classe/dataset

# Função de compressão das imagens
def random_jpeg_compression(img, p=0.5, min_quality=55, max_quality=90):
    """Aplica compressão JPEG em memória com variação aleatória."""
    if random.random() < p:
        buffer = io.BytesIO()
        if img.mode in ('RGBA', 'P'):
            img = img.convert('RGB')
        quality = random.randint(min_quality, max_quality)
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        img = Image.open(buffer)
    return img

# Classe do dataset unificado (SID_Set + AIGI-Holmes + Artifact)
class UnifiedForensicDataset(Dataset):
    def __init__(self, samples, preprocess_fn):
        self.samples = samples
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        amostra = self.samples[idx]
        try:
            if amostra['tipo'] == 'parquet':
                # Leitura do SID-Set
                p_file = pq.ParquetFile(amostra['file_path'])
                rg_table = p_file.read_row_group(amostra['rg_idx'], columns=['image'])
                img_bytes = rg_table['image'][amostra['row_idx']]['bytes'].as_py()
                image = Image.open(io.BytesIO(img_bytes)).convert('RGB')
            else:
                # Leitura direta do disco (Holmes/Artifact)
                image = Image.open(amostra['file_path']).convert('RGB')
                
            # Aplica ruído JPEG e o pré-processador nativo do modelo
            image = random_jpeg_compression(image, p=0.5, min_quality=60, max_quality=95)
            pixel_values = self.preprocess_fn(image)
        except Exception:
            # Fallback seguro para arquivos corrompidos
            pixel_values = torch.zeros((3, 224, 224))
            
        return pixel_values, amostra['label']

Cloning into 'ClipBased-SyntheticImageDetection'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 133 (delta 39), reused 48 (delta 27), pack-reused 58 (from 1)
Receiving objects: 100% (133/133), 597.93 KiB | 22.15 MiB/s, done.
Resolving deltas: 100% (50/50), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
/kaggle/working/ClipBased-SyntheticImageDetection
Ambiente configurado com sucesso! Dispositivo: cuda


## **Mapeamento do AIGI_Holmes e Artifact:**

In [ ]:
print("Mapeando Holmes e Artifact...")
amostras_treino = []
amostras_val = []
inferencia_artifact = []

def realizar_split(lista_caminhos, label, lista_inferencia=None, tipo='image'):
    """
    Fatia a lista embaralhada em conjuntos de Treino e Validação.
    As primeiras 'QTD_TREINO' imagens vão pro treino, o restante compõe a validação.
    """
    for i, path in enumerate(lista_caminhos):
        item = {'tipo': tipo, 'file_path': path, 'label': label}
        if i < QTD_TREINO:
            amostras_treino.append(item)
        elif i < (QTD_TREINO + QTD_VAL):
            amostras_val.append(item)
        elif lista_inferencia is not None:
            lista_inferencia.append(item)


# AIGI Holmes dataset
# Busca todos os arquivos de imagem recursivamente nas pastas de reais e fakes
holmes_real = glob.glob("/kaggle/input/datasets/jvfrostbr/aigi-holmes-dataset-train/0_real/**/*.*", recursive=True)
holmes_fake = glob.glob("/kaggle/input/datasets/jvfrostbr/aigi-holmes-dataset-train/1_fake/**/*.*", recursive=True)

# Coleta uma amostra aleatória respeitando o limite (QTD_TOTAL = 25k) e já divide entre treino/val
realizar_split(random.sample(holmes_real, min(QTD_TREINO + QTD_VAL, len(holmes_real))), 0)
realizar_split(random.sample(holmes_fake, min(QTD_TREINO + QTD_VAL, len(holmes_fake))), 1)


# Artifact dataset 
artifact_dir = '/kaggle/input/datasets/awsaf49/artifact-dataset'
metadata_list = []

# Varredura dos metadados: Lê o CSV de cada uma das 33 subpastas
if os.path.exists(artifact_dir):
    for folder in os.listdir(artifact_dir):
        csv_path = os.path.join(artifact_dir, folder, 'metadata.csv')
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                # Monta o caminho completo do arquivo
                df['full_path'] = os.path.join(artifact_dir, folder) + '/' + df['image_path'].astype(str)
                # Garante o formato binário padrão: 0 para Real, 1 para Fake
                df['target'] = np.where(pd.to_numeric(df['target'], errors='coerce').fillna(1).astype(int) == 0, 0, 1)
                # Guarda a origem (ex: afhq, big_gan) para podermos estratificar depois
                df['source_folder'] = folder
                metadata_list.append(df[['full_path', 'target', 'source_folder']])
            except: pass

if metadata_list:
    df_full_art = pd.concat(metadata_list, ignore_index=True)
    
    # Define quantas imagens devem sair de cada um dos geradores para compor as 30 mil imagens do dataset
    cota = QTD_TOTAL // df_full_art['source_folder'].nunique()
    
    def amostragem_justa(df, cota):
        """ Tenta puxar a cota exata de cada subpasta para garantir diversidade """
        return df.groupby('source_folder', group_keys=False).apply(lambda x: x.sample(n=min(len(x), cota), random_state=SEED))
    
    def completar_cota(df_estratificado, df_original, limite):
        """ Se uma pasta pequena não atingiu a cota, preenche o espaço sorteando do restante da base """
        falta = limite - len(df_estratificado)
        if falta > 0:
            df_sobra = df_original.drop(df_estratificado.index)
            return pd.concat([df_estratificado, df_sobra.sample(n=falta, random_state=SEED)])
        return df_estratificado.sample(n=limite, random_state=SEED)

    # Isola o pool de Reais e Fakes
    df_real_total = df_full_art[df_full_art['target'] == 0]
    df_fake_total = df_full_art[df_full_art['target'] == 1]
    
    # Aplica a amostragem justa garantindo que bata o volume final (QTD_TOTAL)
    df_art_real = completar_cota(amostragem_justa(df_real_total, cota), df_real_total, QTD_TOTAL)
    df_art_fake = completar_cota(amostragem_justa(df_fake_total, cota), df_fake_total, QTD_TOTAL)

    # Extrai as strings de caminho e envia para o split (Treino vs Validação)
    realizar_split(df_art_real['full_path'].tolist(), 0, inferencia_artifact)
    realizar_split(df_art_fake['full_path'].tolist(), 1, inferencia_artifact)
    
print("✅ Holmes e Artifact processados e balanceados.")

Mapeando Holmes e Artifact...


/tmp/ipykernel_22/1449814250.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('source_folder', group_keys=False).apply(lambda x: x.sample(n=min(len(x), cota), random_state=SEED))
/tmp/ipykernel_22/1449814250.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('source_folder', group_keys=False).apply(lambda x: x.sample(n=min(len(x), cota), random_state=SEED))


✅ Holmes e Artifact processados e balanceados.


## **Mapeando o SID_Set:**

In [3]:
print(" Mapeando SID-Set (Parquets) com Amostragem Justa...")

train_dirs = [
    "/kaggle/input/datasets/jvfrostbr/sid-set-train-0-to-99",
    "/kaggle/input/datasets/jvfrostbr/sid-set-train-99-to-199",
    "/kaggle/input/datasets/jvfrostbr/sid-set-train-200-to-248"
]

train_files = []
for d in train_dirs:
    train_files.extend(glob.glob(os.path.join(d, "**/*.parquet"), recursive=True))

train_files = sorted(list(set(train_files)))
cota_por_arquivo = QTD_TOTAL // len(train_files) if train_files else 0

print(f"Total de arquivos Parquet: {len(train_files)}")
print(f"Cota alvo por arquivo: ~{cota_por_arquivo} imagens Reais e {cota_por_arquivo} Fakes.")

sid_real_selecionados = []
sid_fake_selecionados = []
sobras_real = []
sobras_fake = []

# Extrai a cota justa de cada arquivo individualmente
for f in tqdm(train_files, desc="Extraindo cotas dos Parquets"):
    reais_neste_arquivo = []
    fakes_neste_arquivo = []
    
    try:
        p_file = pq.ParquetFile(f)
        for rg_idx in range(p_file.num_row_groups):
            labels = p_file.read_row_group(rg_idx, columns=['label'])['label'].to_pylist()
            for row_idx, label in enumerate(labels):
                item = {'tipo': 'parquet', 'file_path': f, 'rg_idx': rg_idx, 'row_idx': row_idx, 'label': label}
                if label == 0:
                    reais_neste_arquivo.append(item)
                elif label == 1:
                    fakes_neste_arquivo.append(item)
                    
        # Embaralha o conteúdo do arquivo para não pegar sempre do começo do Parquet
        random.shuffle(reais_neste_arquivo)
        random.shuffle(fakes_neste_arquivo)
        
        # Pega a cota exata (ou o limite do arquivo, se for menor que a cota)
        sid_real_selecionados.extend(reais_neste_arquivo[:cota_por_arquivo])
        sid_fake_selecionados.extend(fakes_neste_arquivo[:cota_por_arquivo])
        
        # O que não entrou na cota vai para as "sobras"
        sobras_real.extend(reais_neste_arquivo[cota_por_arquivo:])
        sobras_fake.extend(fakes_neste_arquivo[cota_por_arquivo:])
        
    except Exception:
        continue

#  Preenche o que faltou para bater os 25.000 usando as sobras
falta_real = QTD_TOTAL - len(sid_real_selecionados)
if falta_real > 0:
    random.shuffle(sobras_real)
    sid_real_selecionados.extend(sobras_real[:falta_real])

falta_fake = QTD_TOTAL - len(sid_fake_selecionados)
if falta_fake > 0:
    random.shuffle(sobras_fake)
    sid_fake_selecionados.extend(sobras_fake[:falta_fake])

# Embaralhamento das listas finais do SID-Set antes do corte
random.shuffle(sid_real_selecionados)
random.shuffle(sid_fake_selecionados)

print(f"✅ SID-Set mapeado: {len(sid_real_selecionados)} Reais e {len(sid_fake_selecionados)} Fakes extraídas.")

 Mapeando SID-Set (Parquets) com Amostragem Justa...
Total de arquivos Parquet: 249
Cota alvo por arquivo: ~120 imagens Reais e 120 Fakes.


Extraindo cotas dos Parquets: 100%|██████████| 249/249 [00:09<00:00, 25.70it/s]


✅ SID-Set mapeado: 30000 Reais e 30000 Fakes extraídas.


## **Unificando os 3 datasets em um conjunto de treino e validação:**

In [4]:
# Lista exclusiva para segurar os metadados de teste do SID-Set
inferencia_sidset = []

# Distribução entre as listas de Treino, Validação e Inferência globais (Corte de 23k / 2k / 5k)
for i, item in enumerate(sid_real_selecionados):
    if i < QTD_TREINO:
        amostras_treino.append(item)
    elif i < (QTD_TREINO + QTD_VAL):
        amostras_val.append(item)
    else:
        inferencia_sidset.append(item)

for i, item in enumerate(sid_fake_selecionados):
    if i < QTD_TREINO:
        amostras_treino.append(item)
    elif i < (QTD_TREINO + QTD_VAL):
        amostras_val.append(item)
    else:
        inferencia_sidset.append(item)

# Embaralhamento global obrigatório das listas unificadas (Mistura SID + Artifact + Holmes)
random.shuffle(amostras_treino)
random.shuffle(amostras_val)

print("\n" + "="*50)
print("RESUMO FINAL DO DATASET UNIFICADO")
print("="*50)
print(f"Total de amostras no TREINO: {len(amostras_treino)}")
print(f"Total de amostras na VALIDAÇÃO: {len(amostras_val)}")
print(f"Total para INFERÊNCIA física (Artifact + SID-Set): {len(inferencia_artifact) + len(inferencia_sidset)}")


RESUMO FINAL DO DATASET UNIFICADO
Total de amostras no TREINO: 138000
Total de amostras na VALIDAÇÃO: 12000
Total para INFERÊNCIA física (Artifact + SID-Set): 20000


# **Exportando o dataset de teste do Artifact e SID_Set**

In [ ]:
import zipfile
import io
import shutil

# Caminhos dos arquivos compactados finais no output do Kaggle
zip_treino_path = "/kaggle/working/dataset_treino_estatico.zip"
zip_teste_path = "/kaggle/working/dataset_inferencia_estatico.zip"

print("📦 Iniciando exportação blindada para arquivos ZIP (Evitando limite de arquivos)...")

# Exportação do conjunto de treino e validação (138.000 Imagens)
print("\n🎬 Empacotando Dataset de TREINO e VALIDAÇÃO...")
with zipfile.ZipFile(zip_treino_path, 'w', zipfile.ZIP_STORED) as z_train:
    
    # Processa a lista unificada de treino
    for idx, amostra in enumerate(tqdm(amostras_treino, desc="Treino -> ZIP")):
        subpasta_classe = "0_real" if amostra['label'] == 0 else "1_fake"
        caminho_no_zip = f"treino/{subpasta_classe}/img_{idx}.jpg"
        
        try:
            if amostra['tipo'] == 'parquet':
                p_file = pq.ParquetFile(amostra['file_path'])
                rg_table = p_file.read_row_group(amostra['rg_idx'], columns=['image'])
                img_bytes = rg_table['image'][amostra['row_idx']]['bytes'].as_py()
                
                # Aplica a compressão JPEG padrão do pipeline antes de salvar
                img_pil = Image.open(io.BytesIO(img_bytes)).convert('RGB')
                img_pil = random_jpeg_compression(img_pil, p=0.5, min_quality=60, max_quality=95)
                
                buffer = io.BytesIO()
                img_pil.save(buffer, format="JPEG", quality=95)
                z_train.writestr(caminho_no_zip, buffer.getvalue())
            else:
                # Para Holmes/Artifact, lê do disco, aplica ruído e joga no zip
                img_pil = Image.open(amostra['file_path']).convert('RGB')
                img_pil = random_jpeg_compression(img_pil, p=0.5, min_quality=60, max_quality=95)
                
                buffer = io.BytesIO()
                img_pil.save(buffer, format="JPEG", quality=95)
                z_train.writestr(caminho_no_zip, buffer.getvalue())
        except: pass

    # Processa a lista unificada de validação
    for idx, amostra in enumerate(tqdm(amostras_val, desc="Validação -> ZIP")):
        subpasta_classe = "0_real" if amostra['label'] == 0 else "1_fake"
        caminho_no_zip = f"validacao/{subpasta_classe}/img_{idx}.jpg"
        
        try:
            if amostra['tipo'] == 'parquet':
                p_file = pq.ParquetFile(amostra['file_path'])
                rg_table = p_file.read_row_group(amostra['rg_idx'], columns=['image'])
                img_bytes = rg_table['image'][amostra['row_idx']]['bytes'].as_py()
                img_pil = Image.open(io.BytesIO(img_bytes)).convert('RGB')
                img_pil = random_jpeg_compression(img_pil, p=0.5, min_quality=60, max_quality=95)
                
                buffer = io.BytesIO()
                img_pil.save(buffer, format="JPEG", quality=95)
                z_train.writestr(caminho_no_zip, buffer.getvalue())
            else:
                img_pil = Image.open(amostra['file_path']).convert('RGB')
                img_pil = random_jpeg_compression(img_pil, p=0.5, min_quality=60, max_quality=95)
                
                buffer = io.BytesIO()
                img_pil.save(buffer, format="JPEG", quality=95)
                z_train.writestr(caminho_no_zip, buffer.getvalue())
        except: pass

#  Exportação do conjunto de inferência/teste (20.000 Imagens)
print("\n🎬 Empacotando Dataset de INFERÊNCIA (Artifact + SID-Set)...")
with zipfile.ZipFile(zip_teste_path, 'w', zipfile.ZIP_STORED) as z_test:
    
    def gravar_lote_zip(lista_meta, nome_dataset):
        for idx, amostra in enumerate(tqdm(lista_meta, desc=f"Teste {nome_dataset} -> ZIP")):
            subpasta_classe = "0_real" if amostra['label'] == 0 else "1_fake"
            caminho_no_zip = f"{nome_dataset}/{subpasta_classe}/img_{idx}.jpg"
            
            try:
                if amostra['tipo'] == 'parquet':
                    p_file = pq.ParquetFile(amostra['file_path'])
                    rg_table = p_file.read_row_group(amostra['rg_idx'], columns=['image'])
                    img_bytes = rg_table['image'][amostra['row_idx']]['bytes'].as_py()
                    img_pil = Image.open(io.BytesIO(img_bytes)).convert('RGB')
                    
                    buffer = io.BytesIO()
                    img_pil.save(buffer, format="JPEG", quality=95)
                    z_test.writestr(caminho_no_zip, buffer.getvalue())
                else:
                    img_pil = Image.open(amostra['file_path']).convert('RGB')
                    buffer = io.BytesIO()
                    img_pil.save(buffer, format="JPEG", quality=95)
                    z_test.writestr(caminho_no_zip, buffer.getvalue())
            except: pass

    gravar_lote_zip(inferencia_artifact, "artifact")
    gravar_lote_zip(inferencia_sidset, "sidset")

print("\n" + "="*60)
print(" Os datasets foram gerados e lacrados com sucesso!")
print(f"📁 Treino/Val: {zip_treino_path}")
print(f"📁 Teste Puro: {zip_teste_path}")
print("="*60)

📦 Iniciando exportação blindada para arquivos ZIP (Evitando limite de arquivos)...

🎬 Empacotando Dataset de TREINO e VALIDAÇÃO...


Validação -> ZIP: 100%|██████████| 12000/12000 [31:28<00:00,  6.35it/s]



🎬 Empacotando Dataset de INFERÊNCIA (Artifact + SID-Set)...


Teste sidset -> ZIP: 100%|██████████| 10000/10000 [1:12:36<00:00,  2.30it/s]


 Os datasets foram gerados e lacrados com sucesso!
📁 Treino/Val: /kaggle/working/dataset_treino_estatico.zip
📁 Teste Puro: /kaggle/working/dataset_inferencia_estatico.zip
